# Seção 10 (serializadores com `Annotated`) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`02_serializadores_customizados`](02_serializadores_customizados.ipynb) | `pytz` dispensável; formato sem os parênteses de `(UTC)`; `WrapSerializer` e `return_type` |
| [`04_projeto_solucao_instrutor`](04_projeto_solucao_instrutor.ipynb) | `CustomDate` não lê o próprio formato de saída (ida e volta pelo JSON quebra) |

## Novas

**1. Serializador num `Annotated[list[...]]` recebe a lista, não cada item.**

In [1]:
import json
from datetime import datetime
from typing import Annotated
from pydantic import BaseModel, PlainSerializer
from pydantic_core import PydanticSerializationError


def br_date(value: datetime) -> str:
    return value.strftime("%d/%m/%Y")


WrongDates = Annotated[list[datetime], PlainSerializer(br_date, when_used="json")]
BrDate = Annotated[datetime, PlainSerializer(br_date, return_type=str, when_used="json")]


class WrongReport(BaseModel):
    days: WrongDates


class Report(BaseModel):
    days: list[BrDate]


try:
    WrongReport(days=[datetime(2026, 1, 1)]).model_dump_json()
except PydanticSerializationError as ex:
    print(f"no Annotated da lista: {str(ex).split(': ', 2)[-1][:110]}")
print(f"no Annotated do item: {Report(days=[datetime(2026, 1, 1)]).model_dump_json()}")

no Annotated da lista: Error calling function `br_date`: AttributeError: 'list' object has no attribute 'strftime'
no Annotated do item: {"days":["01/01/2026"]}


A validação passa; o erro só aparece no **dump** (talvez só em produção, na primeira resposta). O tipo com serializador é o do
**elemento**: `list[BrDate]`.

**2. `when_used="json"` não vale para `json.dumps(model_dump())`.**

In [2]:
class Event(BaseModel):
    at: BrDate


event = Event(at=datetime(2026, 1, 1))
print(f"event.model_dump_json(): {event.model_dump_json()}")
print(f"json.dumps(event.model_dump(), default=str): {json.dumps(event.model_dump(), default=str)}")

event.model_dump_json(): {"at":"01/01/2026"}
json.dumps(event.model_dump(), default=str): {"at": "2026-01-01 00:00:00"}


O segundo caminho é comum em código que monta um envelope (`{"data": model.model_dump(), "meta": ...}`) e serializa no fim. O
formato customizado some. Ou o envelope também é um modelo, ou use `model_dump(mode="json")` para montar o `dict`.

**3. Serializador sem `return_type` em `lambda`: o schema herda o tipo original.**

In [3]:
LambdaDate = Annotated[datetime, PlainSerializer(lambda value: value.strftime("%d/%m/%Y"), when_used="json")]


class LambdaEvent(BaseModel):
    at: LambdaDate


print(f"schema de saída: {LambdaEvent.model_json_schema(mode='serialization')['properties']['at']}")
print(f"saída real: {LambdaEvent(at=datetime(2026, 1, 1)).model_dump_json()}")

schema de saída: {'format': 'date-time', 'title': 'At', 'type': 'string'}
saída real: {"at":"01/01/2026"}


`lambda` não tem anotação de retorno, então o schema anuncia `date-time` (ISO) enquanto a API devolve `01/01/2026`. Em
serializadores, função nomeada com `-> str` ou `return_type=str` explícito.

## Veja também

- [`04_aliases_serializacao_desserializacao/99_pegadinhas_e_caveats`](../04_aliases_serializacao_desserializacao/99_pegadinhas_e_caveats.ipynb):
  `@field_serializer` sem `when_used` muda o dump em Python; schema com e sem anotação de retorno.